# Group 30 GPU baseline experiments

기존 Autoencoder / CNN / 50:50 앙상블을 같은 분할로 평가합니다. PatchCore와 AnomalyDINO는 아직 포함하지 않습니다.

1. Google Drive의 `MyDrive/PRS_Group30` 폴더에 `Visual_Defect_Detection_GPU.zip`과 `mvtec_anomaly_detection.tar.xz`를 업로드하세요.
2. Colab에서 GPU 런타임을 선택하고 위에서 아래로 실행하세요.
3. 첫 실행은 bottle 한 종류입니다. 성공한 뒤 CATEGORIES를 늘리세요.

정상 이미지의 20%로 점수·임계값을 보정하고, test는 최종 평가에만 사용합니다. 결과를 본 뒤 설정을 고르면 별도 개발 실험으로 표시해야 합니다. 이 노트북은 아직 Colab GPU에서 검증하지 않았습니다.


In [ ]:
import datetime
import json
import shutil
import subprocess
import sys
import tarfile
import zipfile
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/PRS_Group30')
SOURCE_ZIP = BASE / 'Visual_Defect_Detection_GPU.zip'
DATA_ARCHIVE = BASE / 'mvtec_anomaly_detection.tar.xz'
CATEGORIES = ['bottle']  # 전체: bottle, cable, capsule, carpet, grid, hazelnut, leather, metal_nut, pill, screw, tile, toothbrush, transistor, wood, zipper
EPOCHS = 10
IMAGE_SIZE = 128
BATCH_SIZE = 16
SEED = 42
assert SOURCE_ZIP.is_file(), f'Upload source ZIP: {SOURCE_ZIP}'
assert DATA_ARCHIVE.is_file(), f'Upload dataset: {DATA_ARCHIVE}'


## 코드 설치와 GPU 확인

In [ ]:
PROJECT = Path('/content/Visual_Defect_Detection_GPU')
PROJECT.mkdir(exist_ok=True)
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    destination = PROJECT.parent if all(name.startswith(PROJECT.name + "/") for name in archive.namelist()) else PROJECT
    for name in archive.namelist():
        if not (destination / name).resolve().is_relative_to(destination.resolve()):
            raise ValueError(f'Unsafe ZIP path: {name}')
    archive.extractall(destination)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT) + '[dev]'], check=True)
import torch

print('PyTorch:', torch.__version__)
assert torch.cuda.is_available(), 'GPU 런타임을 선택한 후 다시 실행하세요.'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=PROJECT, check=True)


## 데이터 압축 해제
처음에는 시간이 걸립니다. 이미지 학습은 Drive 대신 Colab 내부 디스크에서 수행합니다.

In [ ]:
LOCAL_ARCHIVE = Path('/content/mvtec_anomaly_detection.tar.xz')
DATA_DIR = Path('/content/mvtec_data')
MARKER = DATA_DIR / '.extraction_complete'
if not MARKER.exists():
    DATA_DIR.mkdir(exist_ok=True)
    shutil.copy2(DATA_ARCHIVE, LOCAL_ARCHIVE)
    with tarfile.open(LOCAL_ARCHIVE) as archive:
        archive.extractall(DATA_DIR, filter='data')
    MARKER.touch()
roots = [DATA_DIR] + [p for p in DATA_DIR.iterdir() if p.is_dir()]
matches = [p for p in roots if (p / 'bottle/train/good').is_dir()]
assert len(matches) == 1, 'Cannot identify MVTec AD dataset root'
DATA_ROOT = matches[0]
for category in CATEGORIES:
    assert (DATA_ROOT / category / 'train/good').is_dir(), category
    assert (DATA_ROOT / category / 'test').is_dir(), category
print('Dataset:', DATA_ROOT)


## 정상 데이터 학습과 최종 평가
각 실행은 고유 폴더에 저장됩니다. 완성된 모델과 분할은 평가 전에 Drive에 보관합니다. 중간에 학습이 끊기면 해당 카테고리 학습을 다시 시작해야 합니다.

In [ ]:
RUN_ID = datetime.datetime.now(datetime.UTC).strftime('%Y%m%d_%H%M%S_%f')
RUN_LOCAL = Path('/content/vdd_runs') / RUN_ID
RUN_DRIVE = BASE / 'results' / RUN_ID
RUN_LOCAL.mkdir(parents=True)
RUN_DRIVE.mkdir(parents=True)

def run(*args):
    subprocess.run([sys.executable, '-m', 'vdd.cli', *map(str, args)], cwd=PROJECT, check=True)

settings = {'categories': CATEGORIES, 'epochs': EPOCHS, 'size': IMAGE_SIZE,
                'batch_size': BATCH_SIZE, 'seed': SEED, 'weight': 0.5, 'quantile': 0.95,
                'device': torch.cuda.get_device_name(0)}
(RUN_DRIVE / 'settings.json').write_text(json.dumps(settings, indent=2))
(RUN_DRIVE / 'environment.txt').write_text(subprocess.check_output(
    [sys.executable, '-m', 'pip', 'freeze'], text=True))
shutil.copy2(SOURCE_ZIP, RUN_DRIVE / SOURCE_ZIP.name)

for category in CATEGORIES:
    folder = RUN_LOCAL / category
    manifest = folder / 'split.json'
    model = folder / 'model'
    report = folder / 'report'
    run('prepare', '--root', DATA_ROOT, '--category', category,
        '--output', manifest, '--seed', SEED, '--validation-fraction', 0.2)
    run('train', '--root', DATA_ROOT, '--manifest', manifest,
        '--output', model, '--epochs', EPOCHS, '--size', IMAGE_SIZE,
        '--batch-size', BATCH_SIZE, '--seed', SEED,
        '--weight', 0.5, '--quantile', 0.95, '--device', 'cuda')
    shutil.copytree(folder, RUN_DRIVE / category, dirs_exist_ok=True)
    run('evaluate', '--root', DATA_ROOT, '--manifest', manifest,
        '--checkpoint', model / 'model.pt', '--output', report, '--device', 'cuda')
    shutil.copytree(folder, RUN_DRIVE / category, dirs_exist_ok=True)
    print(category, json.loads((report / 'metrics.json').read_text())['metrics'])
print('Saved results:', RUN_DRIVE)


## 결과 받기
아래 ZIP을 다운로드해서 이 채팅에 첨부하면 성능 비교와 다음 모델 실험에 사용할 수 있습니다. 데이터셋은 포함하지 않습니다.

In [ ]:
from google.colab import files

result_zip = shutil.make_archive(str(Path('/content') / ('vdd_results_' + RUN_ID)), 'zip', RUN_DRIVE)
files.download(result_zip)
